In [1]:
from dotenv import load_dotenv

_ = load_dotenv()

## Creating subagents

In [2]:
from langchain.tools import tool

@tool
def square_root(x: float) -> float:
    """Calculate the square root of a number."""
    return x ** 0.5

@tool
def square(x: float) -> float:
    """Calculate the square of a number."""
    return x **  2

In [3]:
from langchain.agents import create_agent

# create subagents

subagent_1 = create_agent(
    model='gpt-5-nano',
    tools=[square_root]
)

subagent_2 = create_agent(
    model='gpt-5-nano',
    tools=[square]
)

## Calling subagents

In [ ]:
from langchain.messages import HumanMessage

@tool
def call_subagent_1(x: float) -> float:
    """Call subagent 1 in order to calculate the square root of a number."""
    response = subagent_1.invoke(
        {"messages": [HumanMessage(content=f"Calculate the square root of {x}")]}
    )
    return response["messages"][-1].content

@tool
def call_subagent_2(x: float) -> float:
    """Call subagent 2 in order to calculate the square of a number."""
    response = subagent_2.invoke(
        {"messages": [HumanMessage(content=f"Calculate the square of {x}")]}
    )
    return response["messages"][-1].content

### Creating the main agent

In [6]:
main_agent = create_agent(
    model="gpt-5-nano",
    tools=[call_subagent_1, call_subagent_2],
    system_prompt="""
    You are a helpful assistant who can call subagents to calculate the square root or
    square of a number."""
)

## Test

In [ ]:
question = "What is the square root of 456?"
response = main_agent.invoke({"messages": [HumanMessage(content=question)]})

for message in response["messages"]:
    message.pretty_print()

================================ Human Message =================================

What is the square root of 456?
================================== Ai Message ==================================
Tool Calls:
  call_subagent_1 (call_83RoRtNLBdOiT8xvPuuY7ZXf)
 Call ID: call_83RoRtNLBdOiT8xvPuuY7ZXf
  Args:
    x: 456
================================= Tool Message =================================
Name: call_subagent_1

The square root of 456.0 is approximately 21.354156504062622. 

If you’d like a rounded value: 
- to 10 decimals: 21.3541565041 
- to 5 decimals: 21.35416
================================== Ai Message ==================================

The square root of 456 is approximately 21.354156504062622.

Rounded:
- to 10 decimals: 21.3541565041
- to 5 decimals: 21.35416


: 